# 17 — UCI Multicenter Class-Imbalance Processing

Giữ cố định preprocessing P1 sentinel-aware của Lab 16 và so sánh bốn cách xử lý mất cân bằng trên 920 bệnh nhân thật. Resampling chỉ được thực hiện trong training fold LOCO.

Không dùng synthetic hospital augmentation hoặc threshold tuning trong lab này.

> Đây là nghiên cứu trên dữ liệu công khai, không phải bằng chứng lâm sàng hay công cụ chẩn đoán.

In [ ]:
!pip -q install lightgbm seaborn imbalanced-learn

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
THRESHOLD = 0.50
OUTPUT_DIR = Path('/content/uci_multicenter_class_imbalance_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COLUMNS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
FEATURES = COLUMNS[:-1]
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland': 'processed.cleveland.data', 'hungarian': 'processed.hungarian.data', 'switzerland': 'processed.switzerland.data', 'va': 'processed.va.data'}
EXPECTED_ROWS = {'cleveland': 303, 'hungarian': 294, 'switzerland': 123, 'va': 200}

frames = []
for site, filename in FILES.items():
    part = pd.read_csv(f'{BASE_URL}/{filename}', names=COLUMNS, na_values=['?'], skipinitialspace=True)
    part = part.apply(pd.to_numeric, errors='coerce')
    part['site'] = site
    part['target'] = (part['num'] > 0).astype('int8')
    assert len(part) == EXPECTED_ROWS[site]
    frames.append(part)
df = pd.concat(frames, ignore_index=True)
assert len(df) == 920
df[FEATURES + ['num', 'target', 'site']].to_csv(OUTPUT_DIR / 'uci_multicenter_raw.csv', index=False)
display(pd.crosstab(df['site'], df['target'], margins=True))

## Configurations

- C0: P1 sentinel-aware, không weighting/resampling.
- C1: P1 + `class_weight='balanced'`.
- C2: P1 + RandomOverSampler sau preprocessing.
- C3: P1 + SMOTENC sau fold-local imputation.

P1 đổi `trestbps <= 0` và `chol <= 0` thành missing. Không xóa test site và không dùng test site để fit sampler.

In [ ]:
CONFIGS = {
    'C0_P1_unweighted': {'class_weight': None, 'sampler': None},
    'C1_P1_class_weight': {'class_weight': 'balanced', 'sampler': None},
    'C2_P1_random_over': {'class_weight': None, 'sampler': 'random_over'},
    'C3_P1_smotenc': {'class_weight': None, 'sampler': 'smotenc'},
}

def apply_p1(frame):
    out = frame.copy()
    counts = {}
    for col in ['trestbps', 'chol']:
        mask = out[col].notna() & (out[col] <= 0)
        counts[col] = int(mask.sum())
        out.loc[mask, col] = np.nan
    return out, counts

def preprocessor(add_indicators=True, numeric_features=None):
    numeric_features = numeric_features or NUMERICAL_FEATURES
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=add_indicators)), ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=add_indicators)), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('numeric', numeric, numeric_features), ('categorical', categorical, CATEGORICAL_FEATURES)])

def model(name, weight):
    if name == 'Logistic Regression':
        return LogisticRegression(max_iter=2000, class_weight=weight, random_state=RANDOM_STATE)
    return LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15, min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, class_weight=weight, random_state=RANDOM_STATE, verbosity=-1)

def standard_pipeline(name, config):
    steps = [('preprocessor', preprocessor())]
    if config['sampler'] == 'random_over':
        steps.append(('sampler', RandomOverSampler(random_state=RANDOM_STATE)))
    steps.append(('classifier', model(name, config['class_weight'])))
    return ImbPipeline(steps)

def smotenc_data(X_train_raw, X_test_raw, y_train):
    train, train_counts = apply_p1(X_train_raw)
    test, test_counts = apply_p1(X_test_raw)
    missing_names = [f'{c}__missing' for c in FEATURES]
    train_missing = train[FEATURES].isna().astype('int8'); train_missing.columns = missing_names
    test_missing = test[FEATURES].isna().astype('int8'); test_missing.columns = missing_names
    num_imp = SimpleImputer(strategy='median'); cat_imp = SimpleImputer(strategy='most_frequent')
    train_num = pd.DataFrame(num_imp.fit_transform(train[NUMERICAL_FEATURES]), columns=NUMERICAL_FEATURES, index=train.index)
    test_num = pd.DataFrame(num_imp.transform(test[NUMERICAL_FEATURES]), columns=NUMERICAL_FEATURES, index=test.index)
    train_cat = pd.DataFrame(cat_imp.fit_transform(train[CATEGORICAL_FEATURES]), columns=CATEGORICAL_FEATURES, index=train.index)
    test_cat = pd.DataFrame(cat_imp.transform(test[CATEGORICAL_FEATURES]), columns=CATEGORICAL_FEATURES, index=test.index)
    train_ready = pd.concat([train_num, train_cat, train_missing], axis=1)
    test_ready = pd.concat([test_num, test_cat, test_missing], axis=1)
    cat_indices = list(range(len(NUMERICAL_FEATURES), len(FEATURES)))
    sampler = SMOTENC(categorical_features=cat_indices, k_neighbors=3, random_state=RANDOM_STATE)
    train_resampled, y_resampled = sampler.fit_resample(train_ready, y_train)
    train_resampled = pd.DataFrame(train_resampled, columns=train_ready.columns)
    audit = {'sentinel_train': int(sum(train_counts.values())), 'sentinel_test': int(sum(test_counts.values())), 'rows_before': len(y_train), 'rows_after': len(y_resampled), 'positives_before': int(y_train.sum()), 'positives_after': int(np.sum(y_resampled))}
    return train_resampled, test_ready, y_resampled, NUMERICAL_FEATURES + missing_names, audit

def smotenc_preprocessor(numeric_features):
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    categorical = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('numeric', numeric, numeric_features), ('categorical', categorical, CATEGORICAL_FEATURES)])

def metrics(configuration, name, site, y_true, probability, seconds):
    pred = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'configuration': configuration, 'model': name, 'test_site': site, 'test_rows': len(y_true), 'accuracy': accuracy_score(y_true, pred), 'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0), 'pr_auc': average_precision_score(y_true, probability), 'specificity': tn / (tn + fp) if (tn + fp) else np.nan, 'f1': f1_score(y_true, pred, zero_division=0), 'roc_auc': roc_auc_score(y_true, probability), 'brier': brier_score_loss(y_true, probability), 'false_negatives': int(fn), 'fit_seconds': seconds}

## LOCO evaluation

Mỗi site được khóa hoàn toàn làm test. SMOTENC sử dụng imputation và resampling chỉ trên ba site training.

In [ ]:
records = []; audits = []
for configuration, config in CONFIGS.items():
    for test_site in FILES:
        train_df = df[df['site'] != test_site]; test_df = df[df['site'] == test_site]
        X_train_raw, y_train = train_df[FEATURES], train_df['target']
        X_test_raw, y_test = test_df[FEATURES], test_df['target']
        if config['sampler'] == 'smotenc':
            X_train, X_test, y_fit, numeric_features, audit = smotenc_data(X_train_raw, X_test_raw, y_train)
            audits.append({'configuration': configuration, 'test_site': test_site, **audit})
            for name in ['Logistic Regression', 'LightGBM']:
                pipe = ImbPipeline([('preprocessor', smotenc_preprocessor(numeric_features)), ('classifier', model(name, None))])
                started = time.perf_counter(); pipe.fit(X_train, y_fit); seconds = time.perf_counter() - started
                records.append(metrics(configuration, name, test_site, y_test, pipe.predict_proba(X_test)[:, 1], seconds))
        else:
            X_train, train_counts = apply_p1(X_train_raw); X_test, test_counts = apply_p1(X_test_raw)
            audits.append({'configuration': configuration, 'test_site': test_site, 'sentinel_train': int(sum(train_counts.values())), 'sentinel_test': int(sum(test_counts.values())), 'rows_before': len(y_train), 'rows_after': len(y_train), 'positives_before': int(y_train.sum()), 'positives_after': int(y_train.sum())})
            for name in ['Logistic Regression', 'LightGBM']:
                pipe = standard_pipeline(name, config)
                started = time.perf_counter(); pipe.fit(X_train, y_train); seconds = time.perf_counter() - started
                records.append(metrics(configuration, name, test_site, y_test, pipe.predict_proba(X_test)[:, 1], seconds))
results = pd.DataFrame(records).sort_values(['configuration', 'model', 'test_site']).reset_index(drop=True)
audits = pd.DataFrame(audits)
results.to_csv(OUTPUT_DIR / 'class_imbalance_loco_results.csv', index=False)
audits.to_csv(OUTPUT_DIR / 'class_imbalance_fold_audit.csv', index=False)
display(results.round(4))

In [ ]:
summary = results.groupby(['configuration', 'model']).agg(roc_auc_mean=('roc_auc', 'mean'), roc_auc_std=('roc_auc', 'std'), roc_auc_worst=('roc_auc', 'min'), pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'), recall_std=('recall', 'std'), recall_worst=('recall', 'min'), specificity_mean=('specificity', 'mean'), f1_mean=('f1', 'mean'), brier_mean=('brier', 'mean'), false_negatives_total=('false_negatives', 'sum'), fit_seconds_mean=('fit_seconds', 'mean')).reset_index().sort_values(['model', 'roc_auc_worst', 'recall_worst'], ascending=[True, False, False])
print('CLASS-IMBALANCE PROCESSING SUMMARY')
display(summary.round(6))
summary.round(6).to_csv(OUTPUT_DIR / 'class_imbalance_summary.csv', index=False)
baseline = summary[summary['configuration'] == 'C0_P1_unweighted'].set_index('model')
delta = summary.set_index(['configuration', 'model']).copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'pr_auc_mean', 'recall_mean', 'recall_worst', 'brier_mean', 'false_negatives_total']:
    delta[f'delta_vs_C0_{metric}'] = [row[metric] - baseline.loc[model, metric] for (configuration, model), row in delta.iterrows()]
delta.reset_index().round(6).to_csv(OUTPUT_DIR / 'class_imbalance_delta_vs_unweighted.csv', index=False)
display(delta.reset_index().round(6))

plot_data = summary.pivot(index='configuration', columns='model', values='recall_worst')
plot_data.plot(kind='bar', figsize=(10, 4), ylim=(0.4, 1.0), rot=20, title='Worst-site Recall')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'class_imbalance_worst_site_recall.png', dpi=180, bbox_inches='tight'); plt.show()

run_config = {'dataset_rows': 920, 'validation': 'Leave-One-Center-Out', 'preprocessing': 'P1_sentinel_aware', 'threshold': THRESHOLD, 'random_state': RANDOM_STATE, 'synthetic_data': False, 'configurations': CONFIGS, 'smotenc_k_neighbors': 3, 'source_urls': {site: f'{BASE_URL}/{filename}' for site, filename in FILES.items()}}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_class_imbalance_results', 'zip', OUTPUT_DIR)
print('Saved artifacts:', OUTPUT_DIR, zip_path)
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    print('Không chạy trong Colab; ZIP vẫn nằm tại:', zip_path)

## Cách đọc kết quả

Ưu tiên worst-site Recall, false negatives, PR-AUC, Specificity và Brier cùng với ROC-AUC. Nếu weighting/resampling làm Brier xấu đi, xác suất cần calibration ở lab sau. SMOTENC cần kiểm tra thêm theo từng site và nhiều random seed.